# Inspect the QK and OV circuit

Compare Q, K, and V geometry with cumulative count.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
names=["blocks.1.attn.hook_q","blocks.1.attn.hook_k","blocks.1.attn.hook_v"]
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=names); arrays=[c[n][0,:,1].cpu().numpy() for n in names]; counts=cumulative_counts(tokens,vocab)
for label,a in zip(["Q","K","V"],arrays):
 pc=PCA(n_components=1,random_state=6007).fit_transform(a)[:,0]; print(label,"PC1/count r",float(np.corrcoef(pc,counts)[0,1]))
print("shapes",*[a.shape for a in arrays]); plot_trajectory(shared_pca(arrays),["Q","K","V"],"Layer-1 head 1 Q/K/V geometry","figures/layer1_head1_qkv.png")

Q PC1/count r -0.24146254334308134
K PC1/count r 0.1924691899819373
V PC1/count r 0.0036445480021298606
shapes (150, 32) (150, 32) (150, 32)
